# 🚲 Berlin Street-Level Bicycle Risk & Safe Routing

**Part 2 of 2.** [`01_eda_and_analysis.ipynb`](01_eda_and_analysis.ipynb) cleans the
Unfallatlas data and explores the accidents themselves. This notebook picks up from
its cleaned output and adds a second dataset — Berlin's cycling street network from
OpenStreetMap — to move from *where accidents cluster* to *which streets carry them*.

Up to this point every accident has been a dot on a map. A dot does not know which
street it happened on, so the analysis could only speak in terms of districts and
blurred hotspots. Joining the two datasets lets us ask which streets carry the
heaviest accident history per 100 metres of road, once injuries are weighted by
severity. And because the network is a graph, we can route across it, treating
accident history as a cost to avoid in the same way as distance.

This is not EDA — it is the prototype of a data product. Three questions follow:
does the risk score recover patterns already known to be true, is the ranking
stable, and what can honestly be claimed from it?

## Contents
1. [Load the Cleaned Accident Data](#1-load)
2. [Road Network & Accident Snapping](#2-network)
3. [Segment Risk Scoring](#3-scoring)
4. [Robustness of the Ranking](#4-robustness)
5. [The Highest-Risk Streets](#5-streets)
6. [Risk-Aware Routing](#6-routing)
7. [Limitations](#7-limitations)


---
## 1. Load the Cleaned Accident Data <a id='1-load'></a>

Notebook 01 merges ten annual Unfallatlas releases, filters to Berlin bicycle
accidents, and writes the result to `data/processed/`. Reloading that file rather
than repeating the work keeps one definition of the dataset: duplicating the
cleaning is how two copies of an analysis end up disagreeing.

In [21]:
import pandas as pd

CLEAN = "data/processed/berlin_bike_2018_2025.csv"
df_berlin_rad_final = pd.read_csv(CLEAN)

print(f"{len(df_berlin_rad_final):,} rows x {df_berlin_rad_final.shape[1]} columns")
print(df_berlin_rad_final["light_label"].value_counts().to_string())

# Canary: the verified ULICHTVERH mapping has three categories. A fourth means
# this file was written by the old, incorrect label mapping.
assert "dark_unlit" not in set(df_berlin_rad_final["light_label"]), \
    "Stale labels — re-run 01_eda_and_analysis.ipynb to regenerate the CSV."

37,948 rows x 28 columns
light_label
daylight    30474
darkness     5402
twilight     2072


---
## 2. Road Network & Accident Snapping <a id='2-network'></a>

`osmnx` downloads Berlin's cycling network from OpenStreetMap as a
`networkx.MultiDiGraph`: **nodes** are junctions and dead ends, **edges** are the
street segments between them, carrying `length` in metres, `highway` (road class),
`name` and `oneway`.

`network_type="bike"` keeps cycleways, residential streets and paths while dropping
motorways — the network a cyclist can actually use. A `"drive"` graph would omit
protected bike infrastructure entirely. The graph is cached to
`data/berlin_bike.graphml` so this notebook does not depend on the Overpass API
being reachable.

Each accident is then attached to its nearest segment. Distances are computed in
**EPSG:25833 (UTM 33N)** because distances in degrees are meaningless, and snaps
beyond 25 m are discarded rather than assigned to whatever happens to be closest —
a parallel side street or a park path.

In [22]:
import os
import osmnx as ox
import route_risk as rr

GRAPH_PATH = "data/berlin_bike.graphml"

if os.path.exists(GRAPH_PATH):
    G = ox.load_graphml(GRAPH_PATH)
else:
    G = ox.graph_from_place("Berlin, Germany", network_type="bike")
    os.makedirs("data", exist_ok=True)
    ox.save_graphml(G, GRAPH_PATH)

print(f"{G.number_of_nodes():,} nodes | {G.number_of_edges():,} edges")
assert G.number_of_nodes() > 50_000, "this is not Berlin — still the test grid?"

Gp = ox.project_graph(G, to_crs=rr.BERLIN_CRS)
snapped = rr.snap_accidents(Gp, df_berlin_rad_final, max_snap_dist=25.0)

195,623 nodes | 441,544 edges
Snapped 37,896 / 37,948 accidents within 25 m (99.9%); median offset 0.7 m


---
## 3. Segment Risk Scoring <a id='3-scoring'></a>

Two adjustments turn raw accident counts into a usable score.

**Severity weighting.** A fatality is not equivalent to a graze. Weights follow the
BASt *Unfallkostensätze* — the official German accident cost rates used in
blackspot analysis — which put the economic cost per injured person at roughly
€6,600 slight, €149,000 serious and €1.47 M fatal, i.e. a ratio near **1 : 23 : 222**.

**Shrinkage toward a class prior.** Most of the network has no recorded accident at
all, while a few short segments carry a freak count, so raw accidents-per-100m
would rank a 30 m stub as Berlin's most dangerous street. Each segment is pulled
toward the mean rate for its road class:

$$\text{score} = \frac{\text{observed} + m \cdot \text{prior} \cdot \ell}{\ell + m}$$

with $\ell$ the length in units of 100 m and $m$ (`shrinkage`, default 5) setting
how far the class average is trusted over the individual segment. This is the
empirical Bayes approach standard in road-safety analysis.

Risk is keyed on the **undirected** segment: an accident on a two-way street is
evidence about both directions.

In [23]:
edge_risk = rr.build_edge_risk(Gp, snapped, severity_weights=rr.SEVERITY_KSI)
node_risk = rr.build_node_risk(Gp, snapped, severity_weights=rr.SEVERITY_KSI)
_ = rr.apply_risk_weights(Gp, edge_risk, alpha=2.0, node_risk=node_risk)

print(f"risk weights written to {Gp.number_of_edges():,} edges")

238,972 undirected segments | 8.6% with >=1 accident | risk cap (p95) = 1.54
15,888 intersections carry accidents within 20 m
risk weights written to 441,544 edges


In [24]:
table = rr.compare_routes(Gp, (52.4993, 13.4180), (52.5219, 13.4132))
print(table.to_string(index=False))
print(f"detour {table.attrs['detour_pct']}% | "
      f"risk reduction {table.attrs['risk_reduction_pct']}%")

   route  length_m  risk_exposure  n_segments
shortest      3240         2155.9          73
  safest      3398         1655.2          68
detour 4.9% | risk reduction 23.2%


### 3.1 Validity check: does the scoring recover known patterns?

If the method works, primary and secondary roads should score higher risk per 100 m
than residential streets and cycleways — an ordering well established in the
road-safety literature. This tests the method; it is not itself a finding.

In [25]:
print(edge_risk.groupby("highway")["risk"]
      .agg(["count", "mean", "max"])
      .sort_values("mean", ascending=False)
      .round(3).to_string())

                count   mean     max
highway                             
primary          4784  2.220  16.448
trunk              10  1.485   4.176
secondary       16356  1.277  11.867
tertiary        11881  1.092  15.146
primary_link      140  0.786   7.798
secondary_link    251  0.513   4.167
unclassified     2272  0.350   6.749
residential     69679  0.311   9.397
busway              5  0.205   0.518
cycleway        13441  0.191   9.213
living_street    3745  0.181   7.229
pedestrian        615  0.144   1.746
tertiary_link      67  0.034   0.254
path            13802  0.019   1.759
service         93955  0.016   5.897
bridleway         328  0.002   0.160
track            7641  0.002   1.032


### 3.2 Why junctions are scored separately

Turning and crossing conflicts mean much of the cycling risk sits at intersections
rather than along segments. Edge-only scoring smears a junction's accidents onto
whichever approach arm they snapped to, so junctions are scored separately and
charged as a penalty on entry.

In [26]:
import geopandas as gpd
import numpy as np
import osmnx as ox

pts = snapped[["latitude", "longitude"]]
geom = gpd.GeoSeries(
    gpd.points_from_xy(pts["longitude"], pts["latitude"]), crs="EPSG:4326"
).to_crs(Gp.graph["crs"])

_, node_dist = ox.nearest_nodes(
    Gp, X=geom.x.to_numpy(), Y=geom.y.to_numpy(), return_dist=True
)
node_dist = np.asarray(node_dist)

for radius in (10, 20, 30):
    print(f"within {radius:>2} m of a junction: {(node_dist <= radius).mean():.1%}")

within 10 m of a junction: 60.0%
within 20 m of a junction: 80.8%
within 30 m of a junction: 88.8%


---
## 4. Robustness of the Ranking <a id='4-robustness'></a>

The shrinkage parameter $m$ is a modelling choice, so the ranking should not depend
on it. Below, the top 20 riskiest segments are compared across four values.

High overlap would mean the riskiest streets are a property of the data rather than
of a parameter we picked. Low overlap means segment-level rankings are unstable and
risk should be reported at road-class or corridor level instead.

In [27]:
tops = {}
for m in [1, 2, 5, 10]:
    er = rr.build_edge_risk(
        Gp, snapped, severity_weights=rr.SEVERITY_KSI, shrinkage=m
    )
    tops[m] = list(er.nlargest(20, "risk")["pair"])

baseline = set(tops[5])
for m, top in tops.items():
    print(f"m={m:>2}: top-20 overlap with m=5 = {len(baseline & set(top))}/20")

238,972 undirected segments | 8.6% with >=1 accident | risk cap (p95) = 1.46
238,972 undirected segments | 8.6% with >=1 accident | risk cap (p95) = 1.49
238,972 undirected segments | 8.6% with >=1 accident | risk cap (p95) = 1.54
238,972 undirected segments | 8.6% with >=1 accident | risk cap (p95) = 1.42
m= 1: top-20 overlap with m=5 = 3/20
m= 2: top-20 overlap with m=5 = 4/20
m= 5: top-20 overlap with m=5 = 20/20
m=10: top-20 overlap with m=5 = 14/20


---
## 5. The Highest-Risk Streets <a id='5-streets'></a>

Only segments with at least one recorded accident are drawn. Without that filter the
map would colour streets whose score comes purely from the shrinkage prior — dark
red on a road where nothing ever happened. Folium slows badly past a few thousand
features, so the map shows the top 1,500 segments.

> Saved to `berlin_risk_streets.html`. Interactive folium output does **not** render
> in GitHub's notebook preview — open the HTML file directly.

In [28]:
import risk_map as rm

m = rm.risk_map(Gp, edge_risk, top_n=1500, min_accidents=1)
m

saved berlin_risk_streets.html  (1,500 segments)


In [29]:
print(rm.top_risk_table(edge_risk, Gp, n=20).to_string())

                                      accidents  weighted  length_km  weighted_per_km
name                                                                                 
Weinbergsweg                               68.0     173.0       0.41           417.15
Veteranenstraße                            30.0      72.0       0.35           205.79
Schlesische Straße                         69.0     132.0       0.65           204.40
Kopernikusstraße                           59.0     115.0       0.69           166.68
Alte Schönhauser Straße                    36.0      78.0       0.47           166.59
Konrad-Adenauer-Straße                     14.0      64.0       0.40           160.15
Glogauer Straße                            25.0      67.0       0.43           154.44
Alt-Köpenick                               35.0      70.0       0.47           148.92
Otto-Braun-Straße                         119.0     246.0       1.67           146.98
Warschauer Straße, Warschauer Brücke       48.0     11

---
## 6. Risk-Aware Routing <a id='6-routing'></a>

With a score on every segment, routing becomes an explicit trade-off:

$$\text{cost} = \text{length} \times (1 + \alpha \cdot \text{risk})
              + \text{penalty} \times \text{junction risk}$$

$\alpha$ is the exchange rate between metres and risk — $\alpha = 0$ reproduces the
shortest path, higher values accept longer detours for safer streets. The example
routes Kottbusser Tor to Alexanderplatz.

> **How to report this:** "4.9% longer, 23.5% less historical risk exposure" is
> defensible. "This is the safest route" is not.

In [30]:
table = rr.compare_routes(Gp, (52.4993, 13.4180), (52.5219, 13.4132))
print(table.to_string(index=False))
print(f"detour {table.attrs['detour_pct']}% | "
      f"risk reduction {table.attrs['risk_reduction_pct']}%")

   route  length_m  risk_exposure  n_segments
shortest      3240         2155.9          73
  safest      3398         1655.2          68
detour 4.9% | risk reduction 23.2%


---
## 7. Limitations <a id='7-limitations'></a>

**1. No exposure denominator.** The dataset contains only accidents — no record of
trips that ended safely, no cyclist count per street. A high score partly reflects
how many people cycle there, so the router may prefer empty streets that are not
objectively safer. Note that `path` scores 0.016 against primary's 2.224: zero
crashes on an unlit trail is absence of riders, not safety.

**2. Accidents are double-counted between the link and junction models.** Both
scores are currently built from the full accident set, so a crash near an
intersection is charged twice. Partitioning each accident to exactly one locus is
the fix, and it is outstanding.

**3. Segment rankings are unstable.** Top-20 overlap with the $m=5$ baseline is
3/20 at $m=1$ and 14/20 at $m=10$. Road-class and whole-street aggregates are
robust; a named list of the 20 worst segments is not.

**4. The network is today; the accidents are history.** The OSM graph reflects
current Berlin while accidents span 2018–2025, and the city added protected bike
infrastructure in that window — biasing scores against recently improved streets.

**5. No temporal validation yet.** "Risk reduction" currently means reduced
*historical* exposure. Building risk from 2018–2023 and measuring what share of
2024–2025 crashes fall in the riskiest decile of network length would test whether
past risk predicts future crashes at all.

> **Framing for stakeholders:** this identifies streets with a high recorded
> accident history, normalised for length and weighted for severity. It does not
> predict individual risk and is not a safety guarantee.